# [1] 합성곱 신경망 CNN

In [5]:
# CNN실습시 필수 : 코랩 오른쪽 상단에 -> 연결V 클릭하여 -> 런타임 유형 변경
# T4 GPU( 그래픽 사용 ) 클릭 -> 저장 -> 연결
# https://ufldl.stanford.edu/housenumbers/
# 0) 실습에 필요한 데이터셋( SVHN 칼라의 0부터 9까지의 이미지)
from torchvision import datasets
raw_train = datasets.SVHN( root='./data', split='train', download=True )
raw_test = datasets.SVHN( root='./data', split='test', download=True )


100%|██████████| 182M/182M [00:40<00:00, 4.46MB/s]
100%|██████████| 64.3M/64.3M [00:08<00:00, 7.79MB/s]


In [13]:
# 0) GPU 연산 장치 설정
import torch    # ;cuda' : 엔비디아 그래픽 드라이버 이름
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
# 1) 파이토치의 텐서 타입으로 변경하고 0~ 255(밝기정도) 값들 0 ~ 1 정규화
# 샘플로 2만개만 사용 # 연산속도 위한 cpu가 아닌 gpu 처리
X_train = torch.tensor( raw_train.data[:20000], dtype=torch.float32 ).to(device) / 255.0
Y_train = torch.tensor( raw_train.labels[:20000], dtype=torch.long ).to(device)
X_test = torch.tensor( raw_test.data[:2000], dtype=torch.float32 ).to(device) / 255.0
Y_test = torch.tensor( raw_test.labels[:2000], dtype=torch.long ).to(device)
# 2) 합성곱 신경망 레이어 컨테이너 설정
import torch.nn as nn
model = nn.Sequential(
    # (1) 입력층 : 합성곱 -> 렐루 -> 풀링
    # nn.Conv2d( in_channels=채널수(흑백1칼라3), out_channels=특징/특성 맵/개수 )
    # kernel_size= (필터 3*3) 5(5*5) , padding=패딩깊이(1)
    nn.Conv2d( in_channels=3, out_channels=16, kernel_size=3, padding=1 ),
    nn.ReLU(),
    # 만약에 앞전 합성곱 결과 특성맵 ( 16, 32, 32 ) 가정에 풀링 이휴 -> ( 16, 16, 16 )
    nn.MaxPool2d( kernel_size=2 ),
    # (2) 레이어2: 레이어 추가하여 중요한 특성 추가적으로 추출
    # 중간레이어는 앞전 레이어의 out_channels 특성 수와 일치
    nn.Conv2d( in_channels=16, out_channels=32, kernel_size=3, padding=1 ),
    # (16,16,16) -> (32,8,8)
    nn.ReLU(),nn.MaxPool2d( kernel_size=2 ),
    # (3) 출력/분류기 레이어
    nn.Flatten() , # 32 * 8 * 8 (3차원) -> 2048(!차원)으로 평탄화
    nn.Linear( in_features=32*8*8, out_features=64 ),
    nn.ReLU(),
    nn.Linear( in_features=64, out_features=10 ) # 최종 분류 레이어()
).to(device) # GPU 연산 하라는 설정
# 3) 손실함수 : 다중분류이므로 소프트맥스
crierion = nn.CrossEntropyLoss()
# 4) 옵티마이저 : optimizer = optim.AdamW( model.paremerters(), Ir=학습률 )
import torch.optim as optim
optimizer = optim.AdamW( model.parameters(), lr=0.005 )
# 5) 에포크(학습수) 단위 학습( 순전파 -> 오차계산 -> 역전파 -> 업데이트 )
for epoch in range( 1000 ):
    model.train()               # 1 - 학습모드
    optimizer.zero_grad()       # 2 - 이전기울기 초기화
    logits = model(X_train)     # 3 - 순전파(학습)
    loss = crierion( logits, Y_train ) # 4 - 오차(학습과 정답 차이)
    loss.backward()             # 5 - 역전파(오차 이용한 기울기/미분 가중치/편향 재설정)
    optimizer.step()            # 6 - 가중치 / 편향 업데이트
    if epoch % 10 == 0 :
        print( epoch,":loss: ", loss.item() )
# 6) 모델평가 , model_train() 학습모드 , model_eval() 평가/추론모드
model.eval()
with torch.no_grad():
    test_logits = model(X_test)  # 테스트용 학습하기(기울기 업데이트 안함)
    test_preds = torch.argmax( test_logits, dim=1 ) # 확률중 가장 높은값 추출(소프트맥스)
    # 강아지? 고양이? 사자? [0.6, 0.2.5, 0.1.5]
from sklearn.metrics import accuracy_score # 정확도
# 텐서값들을 cpu로 변환후 넘파이로 변환
print( accuracy_score( Y_test.cpu().numpy(), test_preds.cpu().numpy() ) )

# 7) 모델추론 # [ [공부시간,결석,G1,G2] ]

cuda
0 :loss:  2.308093309402466
10 :loss:  2.2444262504577637
20 :loss:  2.2374627590179443
30 :loss:  2.2333967685699463
40 :loss:  2.224287986755371
50 :loss:  2.1884641647338867
60 :loss:  2.0757558345794678
70 :loss:  1.8767343759536743
80 :loss:  1.5853384733200073
90 :loss:  1.3234349489212036
100 :loss:  1.141207218170166
110 :loss:  0.9811557531356812
120 :loss:  0.8935179710388184
130 :loss:  0.7894183397293091
140 :loss:  0.7210288643836975
150 :loss:  0.667046308517456
160 :loss:  0.6911475658416748
170 :loss:  0.6163836121559143
180 :loss:  0.5726519823074341
190 :loss:  0.5421344041824341
200 :loss:  0.5184219479560852
210 :loss:  0.4987841248512268
220 :loss:  0.47774481773376465
230 :loss:  0.47475966811180115
240 :loss:  0.4569018483161926
250 :loss:  0.4439973533153534
260 :loss:  0.4236985445022583
270 :loss:  0.4109549820423126
280 :loss:  0.4085454046726227
290 :loss:  0.39072269201278687
300 :loss:  0.3763241171836853
310 :loss:  0.3654719591140747
320 :loss:  0.3